# 1: Install Dependencies

In [1]:
!pip install --quiet \
    "transformers>=4.47.0" \
    "peft>=0.12.0" \
    "trl>=0.11.0" \
    "datasets>=2.14.0" \
    "bitsandbytes>=0.43.3" \
    "accelerate>=0.34.2" \
    "huggingface-hub>=0.27.0"

import accelerate
import bitsandbytes
import peft
import transformers
import trl

print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 22.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.7 MB/s eta 0:00:00:00:0100:01
transformers: 5.0.0
peft:         0.19.1
trl:          1.15.0
bitsandbytes: 0.50.2
accelerate:   1.13.0
Installation complete.


# 2: HuggingFace Hub Authentication

In [2]:
import os

from huggingface_hub import login, whoami

# Read HF_TOKEN from Kaggle secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret not found. Add it in the Kaggle Secrets panel.")

login(token=HF_TOKEN, add_to_git_credential=False)
user = whoami()
print(f"Logged in as: {user['name']}")
print("Authentication complete.")

Logged in as: muhasin-code
Authentication complete.


# 3: Load Training Data

In [3]:
import json
from pathlib import Path

data_path = Path("/kaggle/input/datasets/muhammedmuhasink/paperlens-extraction-dataset")

train_path = data_path / "train.jsonl"
val_path = data_path / "val.jsonl"

print(f"Train path exists: {train_path.exists()}")
print(f"Val path exists:   {val_path.exists()}")


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


train_data = load_jsonl(train_path)
val_data = load_jsonl(val_path)

print(f"Loaded {len(train_data)} training examples")
print(f"Loaded {len(val_data)} validation examples")

sample = train_data[0]
for msg in sample["messages"]:
    role = msg["role"]
    preview = msg["content"][:100].replace("\n", " ")
    print(f"  [{role}]: {preview}...")

Train path exists: True
Val path exists:   True
Loaded 2082 training examples
Loaded 233 validation examples
  [system]: You are a research assistant specialized in extracting structured information from machine learning ...
  [user]: Paper Title: Minimally Supervised Learning of Affective Events Using Discourse Relations  Paper Abst...
  [assistant]: {"research_question": "What method does the paper 'Minimally Supervised Learning of Affective Events...


# 3b: Verify Label Distribution (v2 dataset)

In [4]:
# Confirm the attached Kaggle dataset is the relabeled version (non-null % per field)

fields = ["research_question", "method", "datasets", "metrics", "key_finding"]
expected = {
    "research_question": 100.0,
    "method": 23.5,
    "datasets": 24.4,
    "metrics": 36.8,
    "key_finding": 40.6,
}

for name, data in [("train", train_data), ("val", val_data)]:
    parsed, bad = [], 0
    for ex in data:
        try:
            parsed.append(json.loads(ex["messages"][-1]["content"]))
        except json.JSONDecodeError:
            bad += 1
    print(f"{name}: {len(parsed)} parsed, {bad} unparseable assistant messages")
    for f in fields:
        pct = 100 * sum(p.get(f) is not None for p in parsed) / max(len(parsed), 1)
        print(f"  {f:18s} {pct:5.1f}% non-null   (guide: ~{expected[f]}%)")

metrics_pct = (
    100
    * sum(json.loads(ex["messages"][-1]["content"]).get("metrics") is not None for ex in train_data)
    / len(train_data)
)
if metrics_pct > 80:
    print(
        "\nWARNING: metrics is non-null in >80% of train rows -> this looks like the OLD dataset. "
        "Attach the newest dataset version before training."
    )

train: 2082 parsed, 0 unparseable assistant messages
  research_question  100.0% non-null   (guide: ~100.0%)
  method              24.0% non-null   (guide: ~23.5%)
  datasets            24.2% non-null   (guide: ~24.4%)
  metrics             36.9% non-null   (guide: ~36.8%)
  key_finding         40.9% non-null   (guide: ~40.6%)
val: 233 parsed, 0 unparseable assistant messages
  research_question  100.0% non-null   (guide: ~100.0%)
  method              23.2% non-null   (guide: ~23.5%)
  datasets            24.9% non-null   (guide: ~24.4%)
  metrics             41.6% non-null   (guide: ~36.8%)
  key_finding         39.9% non-null   (guide: ~40.6%)


# 4: Load Base Model (4-bit QLoRA)

In [5]:
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Safe to re-run: free any previous model/trainer from GPU memory first
for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit NF4 quantization...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory: allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")
else:
    print("Warning: CUDA not available")

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Model loaded successfully!")

Loading Qwen/Qwen2.5-3B-Instruct with 4-bit NF4 quantization...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU memory: allocated=0.90 GB, reserved=1.15 GB
Model loaded successfully!


# 5: Configure LoRA Adapter

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft.tuners.tuners_utils import BaseTunerLayer

model = prepare_model_for_kbit_training(model)

# Explicit module list instead of "all-linear": in this session "all-linear" also wrapped lm_head,
# which current TRL rejects. These 7 projections are exactly what the v1 run trained
# (36 layers x 7 modules x 2 = 504 tensors, 29,933,568 trainable params).
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}   (v1 run: 29,933,568)")
print(f"Total parameters:     {total:,}")
print(f"Trainable %:          {100 * trainable / total:.2f}%")

head = model.get_output_embeddings()
print(f"lm_head wrapped by adapter: {isinstance(head, BaseTunerLayer)}")
assert not isinstance(
    head, BaseTunerLayer
), "lm_head still has a LoRA adapter - TRL will reject this."
print("LoRA configured successfully!")

Trainable parameters: 29,933,568   (v1 run: 29,933,568)
Total parameters:     1,728,606,208
Trainable %:          1.73%
lm_head wrapped by adapter: False
LoRA configured successfully!


In [7]:
from collections import Counter

import torch

cast_count = 0
for _name, param in model.named_parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)
        cast_count += 1

print(f"Cast {cast_count} trainable parameters from bfloat16 to float32.")

print(Counter(p.dtype for p in model.parameters() if p.requires_grad))

Cast 0 trainable parameters from bfloat16 to float32.
Counter({torch.float32: 504})


In [8]:
extra = [n for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
print("Non-LoRA trainable tensors:", extra)
print("Trainable params:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Non-LoRA trainable tensors: []
Trainable params: 29,933,568


# 6: Configure SFTTrainer (completion-only loss)

In [9]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/kaggle/working/paperlens-extraction-adapter-v3"


def to_prompt_completion(examples):
    """messages = [system, user, assistant] -> prompt (system+user) / completion (assistant).

    TRL applies the chat template itself and, for prompt-completion data,
    computes the loss on the completion tokens only.
    """
    rows = []
    for ex in examples:
        msgs = ex["messages"]
        assert msgs[-1]["role"] == "assistant", "last message must be the assistant answer"
        rows.append({"prompt": msgs[:-1], "completion": [msgs[-1]]})
    return rows


train_dataset = Dataset.from_list(to_prompt_completion(train_data))
val_dataset = Dataset.from_list(to_prompt_completion(val_data))

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset:   {len(val_dataset)} examples")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # transformers v5: float < 1 is a ratio of total steps (replaces warmup_ratio)
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="paged_adamw_32bit",
    max_length=1024,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    completion_only_loss=True,  # loss on the assistant JSON only, not system/user prompt
    loss_type="nll",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("SFTTrainer configured successfully!")
print(
    f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}"
)

Train dataset: 2082 examples
Val dataset:   233 examples


Tokenizing train dataset:   0%|          | 0/2082 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2082 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2082 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2082 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/233 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/233 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/233 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/233 [00:00<?, ? examples/s]

SFTTrainer configured successfully!
Effective batch size: 16


# 6b: Verify Completion-Only Masking

In [ ]:
ex = trainer.train_dataset[0]
ids, labels = ex["input_ids"], ex["labels"]
loss_ids = [t for t, label in zip(ids, labels, strict=False) if label != -100]
print(
    f"Sequence length: {len(ids)} | loss-bearing tokens: {len(loss_ids)} "
    f"({100 * len(loss_ids) / len(ids):.1f}%)"
)
print(tokenizer.decode(loss_ids, skip_special_tokens=False))

n = min(200, len(trainer.train_dataset))
fracs = [
    sum(label != -100 for label in e["labels"]) / len(e["labels"])
    for e in (trainer.train_dataset[j] for j in range(n))
]
print(f"Mean share of tokens with loss over {n} examples: {100 * sum(fracs) / n:.1f}%")

Sequence length: 518 | loss-bearing tokens: 142 (27.4%)
{"research_question": "What method does the paper 'Minimally Supervised Learning of Affective Events Using Discourse Relations' propose? Context: Recognizing affective events that trigger positive or negative sentiment has a wide range of natural language processing applications but remains a challenging problem mainly because the polarity of an event is not necessarily predictable from its constituent words", "method": "is simple and only requires a very small seed lexicon and a large raw corpus. Our experiments using Japanese data show that our method learns affective events effectively without manually labeled data. It also improves supervised learning results when labeled data are small.", "datasets": null, "metrics": null, "key_finding": null}<|im_end|>

Mean share of tokens with loss over 200 examples: 13.5%


# 7: Train

In [11]:
import time
from collections import Counter

# Force adapter weights to float32 right before training — do this here,
# in the same cell as trainer.train(), so nothing upstream (Cell 5/6 reruns,
# or SFTTrainer's own init) can silently revert it first.
cast_count = 0
for _name, param in model.named_parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)
        cast_count += 1

print(f"Cast {cast_count} trainable parameters to float32.")
dtype_counts = Counter(p.dtype for p in model.parameters() if p.requires_grad)
print("Trainable dtypes:", dtype_counts)

if any(dt != torch.float32 for dt in dtype_counts):
    raise RuntimeError("Non-float32 trainable parameters remain — do not proceed to training.")

print("\nStarting training...")
print(f"Training samples:     {len(train_dataset)}")
print(f"Validation samples:   {len(val_dataset)}")
print(
    f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}"
)
print(f"Epochs:               {sft_config.num_train_epochs}")

start_time = time.time()
train_result = trainer.train()
elapsed_min = (time.time() - start_time) / 60

print(f"\nTraining completed in {elapsed_min:.1f} minutes")
print(f"Total training steps: {trainer.state.global_step}")
print(f"Final train loss: {train_result.metrics.get('train_loss', 'N/A')}")

eval_entries = [log for log in trainer.state.log_history if "eval_loss" in log]
if eval_entries:
    last_eval = eval_entries[-1]
    print(f"Last eval loss: {last_eval['eval_loss']:.4f} (epoch {last_eval.get('epoch', '?')})")
else:
    print("No eval_loss recorded in log history.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Cast 504 trainable parameters to float32.
Trainable dtypes: Counter({torch.float32: 504})

Starting training...
Training samples:     2082
Validation samples:   233
Effective batch size: 16
Epochs:               3


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.043595,0.043789,0.034639,0.988071,1768692.000000
2,0.025659,0.032245,0.024767,0.990764,3537384.000000
3,0.015562,0.029973,0.021481,0.991968,5306076.000000



Training completed in 182.2 minutes
Total training steps: 387
Final train loss: 0.0566423960049331
Last eval loss: 0.0300 (epoch 3.0)


# 8: Save Adapter Locally

In [12]:
print(f"Saving adapter to {OUTPUT_DIR}/...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verify saved files
saved_files = os.listdir(OUTPUT_DIR)
print(f"Saved files: {saved_files}")

# Check adapter files — PEFT defaults to safetensors now, not .bin,
# so check for either rather than assuming .bin (the milestone doc's
# template only checks .bin, which would wrongly report "missing" here)
adapter_config_exists = os.path.exists(f"{OUTPUT_DIR}/adapter_config.json")
adapter_weights_exists = os.path.exists(
    f"{OUTPUT_DIR}/adapter_model.safetensors"
) or os.path.exists(f"{OUTPUT_DIR}/adapter_model.bin")
print(f"adapter_config.json present: {adapter_config_exists}")
print(f"adapter weights present:     {adapter_weights_exists}")

assert (
    adapter_config_exists and adapter_weights_exists
), "Adapter files missing — check the trainer.save_model() output above for errors."

Saving adapter to /kaggle/working/paperlens-extraction-adapter-v3/...
Saved files: ['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'checkpoint-129', 'tokenizer_config.json', 'checkpoint-258', 'README.md', 'training_args.bin', 'checkpoint-387', 'chat_template.jinja']
adapter_config.json present: True
adapter weights present:     True


# 9: Push Adapter to Hugging Face Hub

In [14]:
from huggingface_hub import HfApi

# v2 repo. Do NOT reuse the v1 name: create_repo(exist_ok=True) + upload_folder
# would silently overwrite the v1 adapter that we still need for comparison.
REPO_NAME = "paperlens-qwen2.5-3b-extraction-v3"  # resolved against your authenticated account

api = HfApi()
repo_url = api.create_repo(
    repo_id=REPO_NAME,
    token=HF_TOKEN,
    exist_ok=True,
    repo_type="model",
)
HF_REPO_ID = repo_url.repo_id  # full "username/repo-name" as HF resolved it
print(f"Repository ready: {HF_REPO_ID}")
print(f"URL: {repo_url}")

# Upload only the adapter + tokenizer files — exclude the per-epoch
# checkpoint-*/ subfolders (full optimizer/scheduler state, redundant
# weight copies) so we stay near the ~100-200MB adapter footprint
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
    ignore_patterns=["checkpoint-*", "training_args.bin"],
)
print("Adapter pushed to HuggingFace Hub!")

files = api.list_repo_files(repo_id=HF_REPO_ID, token=HF_TOKEN, repo_type="model")
print(f"Hub files: {files}")

Repository ready: muhasin-code/paperlens-qwen2.5-3b-extraction-v3
URL: https://huggingface.co/muhasin-code/paperlens-qwen2.5-3b-extraction-v3


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Adapter pushed to HuggingFace Hub!
Hub files: ['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


# 10: Smoke Test

In [15]:
import gc
import re

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Freeing training-time model from GPU memory...")
del model, trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading base model + adapter fresh from the Hub for the smoke test...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

smoke_model = PeftModel.from_pretrained(base_model, HF_REPO_ID)
smoke_model.eval()
print("Adapter loaded from Hub successfully.\n")

# Real held-out validation examples rather than placeholder text
NUM_SMOKE_TESTS = 3
samples = val_data[:NUM_SMOKE_TESTS]

results = []
for i, sample in enumerate(samples):
    messages = sample["messages"]
    prompt_messages = [m for m in messages if m["role"] != "assistant"]
    ground_truth_raw = next((m["content"] for m in messages if m["role"] == "assistant"), None)

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(smoke_model.device)

    with torch.no_grad():
        output_ids = smoke_model.generate(**inputs, max_new_tokens=256, do_sample=False)

    generated_text = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    ).strip()

    json_match = re.search(r"\{.*\}", generated_text, re.DOTALL)
    parsed = None
    valid_json = False
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            valid_json = True
        except json.JSONDecodeError:
            pass

    ground_truth = json.loads(ground_truth_raw) if ground_truth_raw else None

    print(f"--- Test {i+1} ---")
    print(f"Valid JSON: {valid_json}")
    if valid_json:
        print(f"Fields present: {list(parsed.keys())}")
        if ground_truth:
            matches = {k: (parsed.get(k) == ground_truth.get(k)) for k in ground_truth}
            print(f"Field-level match vs ground truth: {matches}")
    else:
        print(f"Raw output (first 200 chars): {generated_text[:200]}")
    print()

    results.append({"index": i, "valid_json": valid_json, "parsed": parsed})

print("=" * 50)
print("Smoke Test Summary:")
print(f"  Total tests: {len(results)}")
print(f"  Valid JSON: {sum(r['valid_json'] for r in results)}/{len(results)}")

Freeing training-time model from GPU memory...
Loading base model + adapter fresh from the Hub for the smoke test...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Adapter loaded from Hub successfully.

--- Test 1 ---
Valid JSON: True
Fields present: ['research_question', 'method', 'datasets', 'metrics', 'key_finding']
Field-level match vs ground truth: {'research_question': True, 'method': False, 'datasets': True, 'metrics': True, 'key_finding': True}

--- Test 2 ---
Valid JSON: True
Fields present: ['research_question', 'method', 'datasets', 'metrics', 'key_finding']
Field-level match vs ground truth: {'research_question': True, 'method': True, 'datasets': True, 'metrics': True, 'key_finding': True}

--- Test 3 ---
Valid JSON: True
Fields present: ['research_question', 'method', 'datasets', 'metrics', 'key_finding']
Field-level match vs ground truth: {'research_question': True, 'method': True, 'datasets': True, 'metrics': True, 'key_finding': False}

Smoke Test Summary:
  Total tests: 3
  Valid JSON: 3/3
